# 01 · Load and Clean

**Goal:** load the raw Olist CSVs into a SQLite database, check data quality,
and build one clean, order-level analysis table for the rest of the project.

Data: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
(CC BY-NC-SA 4.0). Download the CSVs into `data/` before running this notebook.

In [4]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import empiricaldist

In [5]:
DATA_DIR = Path("../data")
DB_PATH = DATA_DIR / "olist.db"

DB_PATH.unlink(missing_ok=True)   # rebuild from scratch
conn = sqlite3.connect(DB_PATH)

csv_paths = sorted(DATA_DIR.glob("*.csv"))
print(f"Found {len(csv_paths)} CSV files")

Found 9 CSV files


In [6]:
def table_name_from(path: Path) -> str:
    """olist_orders_dataset.csv -> orders"""
    return path.stem.replace("olist_", "").replace("_dataset", "")

for path in csv_paths:
    df = pd.read_csv(path)
    df.to_sql(table_name_from(path), conn, if_exists="replace", index=False)

print("Loaded tables:", [table_name_from(p) for p in csv_paths])

Loaded tables: ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'product_category', 'products', 'sellers']


### Verify the load
Compare each table's SQL row count with the raw line count of its CSV file.
A mismatch means the file has line breaks *inside* quoted fields.

In [7]:
print(f"{'TABLE':<35} {'SQL ROWS':>10} {'FILE LINES':>11}")
print("")
for path in csv_paths:
    table = table_name_from(path)
    sql_rows = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn)["n"].iloc[0]
    with open(path, encoding="utf-8") as f:
        file_lines = sum(1 for _ in f) - 1      # minus header
    flag = "  <-- mismatch" if sql_rows != file_lines else ""
    print(f"{table:<35} {sql_rows:>10,} {file_lines:>11,}{flag}")

TABLE                                 SQL ROWS  FILE LINES

customers                               99,441      99,441
geolocation                          1,000,163   1,000,163
order_items                            112,650     112,650
order_payments                         103,886     103,886
order_reviews                           99,224     104,719  <-- mismatch
orders                                  99,441      99,441
product_category                            71          71
products                                32,951      32,951
sellers                                  3,095       3,095


In [8]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Table in olist.db:")
for table in tables:
    query = f"SELECT count(*) FROM {table[0]};"
    row = cursor.execute(query).fetchone()
    print(f"{table[0]:<35} has {row[0]} rows")

Table in olist.db:
customers                           has 99441 rows
geolocation                         has 1000163 rows
order_items                         has 112650 rows
order_payments                      has 103886 rows
order_reviews                       has 99224 rows
orders                              has 99441 rows
product_category                    has 71 rows
products                            has 32951 rows
sellers                             has 3095 rows


All tables match their files except `order_reviews` (99,224 rows vs. 104,719 lines):
review comments contain line breaks, so one row can span several lines.
The SQL row count is the correct one.

### 2.Data Quality Check

`order_reviews` has 99,224 rows but `orders` has 99,441. Before joining,
check whether any order has more than one review (which would duplicate
orders in the join).

In [9]:
query = """
SELECT 
    COUNT(*) AS total_reviews,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(*) - COUNT(DISTINCT order_id) AS gap,
    (SELECT COUNT(*) FROM (
        SELECT order_id
        FROM order_reviews
        GROUP BY order_id
        HAVING COUNT(*) > 1
    ) AS multiple_review_orders) AS orders_with_mreviews
    
FROM order_reviews
"""
result = pd.read_sql_query(query, conn)
print(f'There are {result["orders_with_mreviews"].iloc[0]} orders with multiple reviews.')
result

There are 547 orders with multiple reviews.


,total_reviews,total_orders,gap,orders_with_mreviews
0,99224,98673,551,547


List of Orders with multiple reviews

In [10]:
query = """
SELECT 
    order_id, 
    COUNT(*) AS total_reviews
FROM order_reviews
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY total_reviews DESC;
"""
result = pd.read_sql_query(query, conn)
result.head(5)

,order_id,total_reviews
0,df56136b8031ecd28e200bb18e6ddb2e,3
1,c88b1d1b157a9999ce368f218a407141,3
2,8e17072ec97ce29f0e1f111e598b0c85,3
3,03c939fd7fd3b38f8485a0f95798f1f6,3
4,ffaabba06c9d293a3c614e0515ddbabc,2


There are 4 order has 3 reviews and the rest 543 has 2 reviews.

In [11]:
query = """
WITH M_REVIEWS AS (
SELECT 
    order_id, 
    COUNT(*) AS total_reviews
FROM order_reviews
GROUP BY order_id
HAVING COUNT(*) > 1)

, lag AS (
SELECT order_id, review_creation_date, review_score,lag(review_score) over (partition by order_id order by review_creation_date) as previous_review_score
FROM order_reviews
WHERE order_id IN (SELECT order_id FROM M_REVIEWS)
ORDER BY order_id, review_creation_date)

select *, review_score - previous_review_score as score_change from lag;

"""
result = pd.read_sql_query(query, conn)
result.head(10)

,order_id,review_creation_date,review_score,previous_review_score,score_change
0,0035246a40f520710769010f752e7507,2017-08-25 00:00:00,5,NaN,NaN
1,0035246a40f520710769010f752e7507,2017-08-29 00:00:00,5,5.0,0.0
2,013056cfe49763c6f66bda03396c5ee3,2018-02-22 00:00:00,5,NaN,NaN
3,013056cfe49763c6f66bda03396c5ee3,2018-03-04 00:00:00,4,5.0,-1.0
4,0176a6846bcb3b0d3aa3116a9a768597,2017-12-30 00:00:00,5,NaN,NaN
5,0176a6846bcb3b0d3aa3116a9a768597,2017-12-30 00:00:00,5,5.0,0.0
6,02355020fd0a40a0d56df9f6ff060413,2018-03-21 00:00:00,3,NaN,NaN
7,02355020fd0a40a0d56df9f6ff060413,2018-03-29 00:00:00,1,3.0,-2.0
8,029863af4b968de1e5d6a82782e662f5,2017-07-14 00:00:00,5,NaN,NaN
9,029863af4b968de1e5d6a82782e662f5,2017-07-19 00:00:00,4,5.0,-1.0


In [12]:
query = """
WITH multi AS (
    SELECT order_id
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
),
with_prev AS (
    SELECT
        order_id,
        review_score,
        LAG(review_score) OVER (
            PARTITION BY order_id
            ORDER BY review_answer_timestamp
        ) AS prev_score
    FROM order_reviews
    WHERE order_id IN (SELECT order_id FROM multi)
)
SELECT
    COUNT(*)                            AS review_pairs,
    SUM(review_score <  prev_score)     AS later_lower,
    SUM(review_score =  prev_score)     AS same,
    SUM(review_score >  prev_score)     AS later_higher,
    ROUND(AVG(review_score - prev_score), 2) AS avg_change
FROM with_prev
WHERE prev_score IS NOT NULL;

"""
result = pd.read_sql_query(query, conn)
result


,review_pairs,later_lower,same,later_higher,avg_change
0,551,114,348,89,-0.15


In [13]:
query = """
SELECT 
    count(*) AS total_order,
    (select count(distinct order_id) from order_reviews) AS actual_reviewed_orders,
    count(*) - (select count(distinct order_id) from order_reviews) AS difference

FROM orders


"""
result = pd.read_sql_query(query, conn)
print(f'There are {result["difference"].iloc[0]} order got no review.')
result

There are 768 order got no review.


,total_order,actual_reviewed_orders,difference
0,99441,98673,768


**Finding:** of 551 consecutive review pairs, 348 (63%) kept the same score,
114 (21%) dropped and 89 (16%) rose (average change −0.15 points).
Most duplicate reviews agree; when they differ, drops are slightly more common.

**Decision:** keep only the **latest** review per order (by `review_answer_timestamp`)
as the customer's final opinion. This affects fewer than 0.3% of orders, so the
choice does not materially change later results. Orders without a review are kept
for delivery analysis but excluded from review-score tests (H1).

In [22]:
query = """
SELECT * FROM order_reviews
where order_id ='013056cfe49763c6f66bda03396c5ee3';


"""
result = pd.read_sql_query(query, conn)
result


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,None,None,2018-02-22 00:00:00,2018-02-23 12:12:30
1,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,None,None,2018-03-04 00:00:00,2018-03-05 17:02:00


In [24]:
query = """
SELECT review_id, order_id, review_score, review_comment_title,
       review_comment_message, max(date(review_creation_date)) AS latest_review_date,
       review_answer_timestamp FROM order_reviews
where order_id ='013056cfe49763c6f66bda03396c5ee3'
GROUP BY review_id, order_id, review_score, review_comment_title,
         review_comment_message, review_answer_timestamp;

"""
result = pd.read_sql_query(query, conn)
result


,review_id,order_id,review_score,review_comment_title,review_comment_message,latest_review_date,review_answer_timestamp
0,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,None,None,2018-03-04,2018-03-05 17:02:00
1,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,None,None,2018-02-22,2018-02-23 12:12:30
